# Week 2 — Part 2: Baseline Regression Modeling
### Steel Industry Energy Consumption Dataset

**Objective:** Use the engineered dataset produced in `week2_eda.ipynb` to train
multiple regression models, evaluate them properly, and establish a solid
baseline for predicting `Usage_kWh`.


**Structure of this notebook:**
1. Load engineered dataset
2. Drop leakage columns & the raw date column
3. Encode categorical columns (documented choice)
4. Train/test split (80/20, `random_state=42`)
5. Train 4 models: Linear Regression, Ridge, Decision Tree, Random Forest
6. Evaluate: MAE, RMSE, R² on test set
7. 5-fold cross-validation (mean RMSE)
8. Bar chart: test RMSE comparison across models
9. Scatter plot: Predicted vs Actual for the best model
10. Model Selection write-up


## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42
DATA_PATH = "../Data/Steel_industry_data_engineered.csv"


ModuleNotFoundError: No module named 'sklearn'

## 2. Load Engineered Dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()


## 3. Drop Leaky / Non-Feature Columns

We remove:
- `date` — raw timestamp, already decomposed into `Hour`, `DayOfWeek`, `Month`,
  `Is_Weekend` in Part 1, so the raw string is redundant for modeling.
- `High_Usage_Flag` — this was engineered directly from `Usage_kWh` (whether a
  row is above the 75th percentile), so keeping it would leak the target
  straight into the features and produce an artificially perfect model.
- `NSM` — kept as a feature (it's a legitimate time-of-day signal available
  before the fact, not derived from the target), but we double check nothing
  else derived from `Usage_kWh` remains.

Everything else — the raw sensor readings, power factor, reactive power, CO2,
load type, and the calendar features — are valid predictors.

In [ ]:
leak_cols = ["date", "High_Usage_Flag"]
target_col = "Usage_kWh"

model_df = df.drop(columns=leak_cols)
print("Columns going into modeling:")
print(list(model_df.columns))


## 4. Encode Categorical Columns

**Categorical columns:** `Load_Type`, `DayOfWeek`.

(Note: the raw dataset's original `Day_of_week` and `WeekStatus` columns were
already dropped in `week2_eda.ipynb` because they duplicated the
`date`-derived `DayOfWeek` and `Is_Weekend` columns — see the EDA notebook's
"redundant columns" note. `Is_Weekend` is already numeric (0/1) so it needs no
further encoding.)

**Encoding choice: One-Hot Encoding** (via `pd.get_dummies`) for both,
rather than label encoding. Reasoning:
- Neither category has a natural ordinal relationship (e.g. `Load_Type` =
  Light/Medium/Maximum has an implied order in *name* but the underlying
  relationship with `Usage_kWh` isn't guaranteed to be linear in that order,
  and `DayOfWeek` has no meaningful order at all for a linear/tree-based
  model to exploit).
- Label encoding would impose a false numeric ordering (e.g. Friday=4 >
  Monday=0) that linear models in particular would incorrectly treat as
  having magnitude/order, biasing coefficients.
- The cardinality of each categorical column is low (3 and 7 categories), so
  one-hot encoding doesn't blow up dimensionality.
- We use `drop_first=True` to avoid the dummy variable trap (perfect
  multicollinearity) for the linear/ridge models.

In [ ]:
categorical_cols = ["Load_Type", "DayOfWeek"]
print("Categorical columns and their unique values:")
for c in categorical_cols:
    print(f"  {c}: {model_df[c].unique().tolist()}")

model_df = pd.get_dummies(model_df, columns=categorical_cols, drop_first=True)
print("\nShape after one-hot encoding:", model_df.shape)
model_df.head()


## 5. Train/Test Split (80/20)

In [ ]:
X = model_df.drop(columns=[target_col])
y = model_df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


## 6. Train 4 Models

- **Linear Regression** — simplest baseline, no regularization.
- **Ridge Regression** — linear model with L2 regularization to reduce
  variance / handle multicollinearity between the reactive power / power
  factor features.
- **Decision Tree Regressor** — captures non-linear relationships and
  interactions without needing feature scaling.
- **Random Forest Regressor** — ensemble of trees, typically reduces
  overfitting relative to a single decision tree and often gives the
  strongest baseline performance.

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeRegressor(random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
}

trained_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model
    print(f"Trained: {name}")


## 7. Evaluate on Test Set — MAE, RMSE, R²

In [ ]:
results = []

for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    results.append({"Model": name, "MAE": mae, "RMSE": rmse, "R2": r2})
    print(f"{name:20s} | MAE: {mae:8.3f} | RMSE: {rmse:8.3f} | R2: {r2:.4f}")

results_df = pd.DataFrame(results).set_index("Model")
results_df


## 8. 5-Fold Cross-Validation (Mean RMSE)

Cross-validation gives a more robust estimate of generalization performance
than a single train/test split, and lets us compare test RMSE against CV RMSE
to check for overfitting.

In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = {}

for name, model in models.items():
    # cross_val_score maximizes by default, so use neg_root_mean_squared_error and negate
    scores = cross_val_score(model, X, y, cv=kfold, scoring="neg_root_mean_squared_error", n_jobs=-1)
    rmse_scores = -scores
    cv_results[name] = rmse_scores
    print(f"{name:20s} | CV RMSE (mean +/- std): {rmse_scores.mean():.3f} +/- {rmse_scores.std():.3f}")

cv_summary = pd.DataFrame({name: scores for name, scores in cv_results.items()})
cv_mean_rmse = cv_summary.mean().sort_values()
print("\nMean CV RMSE (sorted, best first):")
print(cv_mean_rmse)


## 9. Bar Chart — Test RMSE Comparison Across Models

In [ ]:
plt.figure(figsize=(8, 5))
order = results_df["RMSE"].sort_values().index
bars = plt.bar(order, results_df.loc[order, "RMSE"], color=sns.color_palette("viridis", len(order)))
plt.title("Test RMSE Comparison Across Models")
plt.xlabel("Model")
plt.ylabel("RMSE (kWh)")
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height, f"{height:.2f}", ha="center", va="bottom")
plt.tight_layout()
plt.show()


## 10. Predicted vs Actual — Best Model

We pick the best model by lowest test RMSE and visualize how well its
predictions track the actual values.

In [ ]:
best_model_name = results_df["RMSE"].idxmin()
best_model = trained_models[best_model_name]
y_pred_best = best_model.predict(X_test)

print(f"Best model (lowest test RMSE): {best_model_name}")

plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred_best, alpha=0.4, color="teal", edgecolor="none")
min_val = min(y_test.min(), y_pred_best.min())
max_val = max(y_test.max(), y_pred_best.max())
plt.plot([min_val, max_val], [min_val, max_val], color="red", linestyle="--", label="Perfect Prediction")
plt.title(f"Predicted vs Actual Usage_kWh — {best_model_name}")
plt.xlabel("Actual Usage (kWh)")
plt.ylabel("Predicted Usage (kWh)")
plt.legend()
plt.tight_layout()
plt.show()


## 11. Model Selection

*(This is a template — fill in the actual numbers your run produces. The
placeholders below reference the variables computed above so you can print
them directly if you prefer an auto-generated version; the markdown here is
written for a typical outcome on this dataset and should be adjusted to match
your printed results.)*

### Summary Table

| Model | Test MAE | Test RMSE | Test R² | CV Mean RMSE |
|---|---|---|---|---|
| Linear Regression | *see output* | *see output* | *see output* | *see output* |
| Ridge Regression | *see output* | *see output* | *see output* | *see output* |
| Decision Tree | *see output* | *see output* | *see output* | *see output* |
| Random Forest | *see output* | *see output* | *see output* | *see output* |

### Which model performed best

The **Random Forest Regressor** typically achieves the lowest test RMSE and
highest R² among the four models on this dataset. This is expected: energy
usage depends on non-linear interactions between load type, time-of-day, and
electrical readings (reactive power, power factor) that a linear model cannot
capture, while a single Decision Tree tends to overfit the training data. The
Random Forest averages many trees, which reduces variance and generally
generalizes better than a lone tree.

### Signs of overfitting

- The **Decision Tree** typically shows a large gap between training
  performance (near-perfect fit) and test/CV RMSE — a classic overfitting
  signature, since an unconstrained tree can memorize training rows.
- The **Random Forest** shows a much smaller train/test gap because
  averaging across trees reduces variance, though some gap still remains.
- **Linear Regression** and **Ridge** show similar train and test error
  (low variance) but usually have higher absolute error than the tree-based
  models (higher bias) — they underfit the non-linear structure in the data.
- Cross-validation RMSE (5-fold) should be reasonably close to the single
  test-set RMSE for each model; a large gap between CV RMSE and test RMSE
  would suggest the 80/20 split wasn't representative.

### Model carried forward

**Random Forest Regressor** is carried forward as the baseline model for
future weeks, since it offers the best balance of low error and reasonable
generalization (train/test gap) without any hyperparameter tuning yet.
Hyperparameter tuning (e.g. `max_depth`, `n_estimators`, `min_samples_leaf`)
and feature importance analysis are natural next steps to further improve and
interpret this baseline.
